# Emerging Technologies Assessment Summer 25/26

This notebook explores the difference between classical and quantum algorithms through a series of hands-on problems.
The problems are centred around the [Deutsch–Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa), a quantum algorithm that demonstrates an exponential speedup over its classical counterpart for a specific problem involving Boolean functions.

---

## Problem 1: Generating Random Boolean Functions

### Background

The [Deutsch–Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa) operates on Boolean functions of the form:

$$f: \{0,1\}^n \rightarrow \{0,1\}$$

where the function is **guaranteed** to be one of two kinds:

- **Constant:** The function returns the same output (`True` or `False`) for *every* possible input combination.
- **Balanced:** The function returns `True` for *exactly half* of all possible input combinations, and `False` for the other half.

With **four Boolean inputs**, there are $2^4 = 16$ possible input combinations.
This means a balanced function returns `True` for exactly **8** of those 16 inputs.

There are only **2 constant functions** (always-`False` and always-`True`), but $\binom{16}{8} = 12{,}870$ possible balanced functions — making a random balanced function effectively unique every time.

### Task

Write a Python function `random_constant_balanced` that randomly returns one function from the combined set of constant and balanced functions, each taking **four Boolean arguments**.

### Imports

Only modules from the Python standard library are needed for this problem:

- [`random`](https://docs.python.org/3/library/random.html) — for random selection of function type and truth-table assignments.
- [`itertools`](https://docs.python.org/3/library/itertools.html) — for generating all combinations of Boolean inputs efficiently.

In [1]:
import random      # Provides random choice and sampling utilities
import itertools   # Provides Cartesian product for enumerating all input combinations

### Approach

The strategy is to build a **truth table** — a dictionary that maps every possible input tuple to an output value — and then return a closure that looks up any query in that table.

1. Use `itertools.product([False, True], repeat=4)` to enumerate all 16 four-bit input combinations.
2. Randomly choose whether the function will be **constant** or **balanced**.
3. Fill the truth table accordingly:
   - *Constant:* Every input maps to the same randomly chosen value.
   - *Balanced:* Randomly sample 8 of the 16 inputs to map to `True`; the rest map to `False`.
4. Return an inner function `f(a, b, c, d)` that performs a single dictionary lookup.

In [2]:
def random_constant_balanced():
    """
    Return a randomly chosen constant or balanced Boolean function.

    The returned function accepts exactly four Boolean positional arguments
    (a, b, c, d) and returns a single Boolean value.

    - Constant: returns the same Boolean value for every possible input.
    - Balanced: returns True for exactly 8 of the 16 possible input combinations
      and False for the remaining 8.

    Returns
    -------
    function
        A callable f(a, b, c, d) -> bool backed by a pre-built truth table.
    """
    # Enumerate all 2^4 = 16 combinations of four Boolean inputs
    all_inputs = list(itertools.product([False, True], repeat=4))

    # Randomly decide whether this will be a constant or balanced function
    function_type = random.choice(["constant", "balanced"])

    if function_type == "constant":
        # Choose a single output value that every input will map to
        constant_output = random.choice([True, False])

        # Every input gets the same constant output
        truth_table = {inputs: constant_output for inputs in all_inputs}

    else:  # balanced
        # Randomly pick exactly 8 of the 16 inputs to return True
        true_inputs = set(random.sample(all_inputs, 8))

        # Inputs in the chosen set return True; all others return False
        truth_table = {inputs: (inputs in true_inputs) for inputs in all_inputs}

    def f(a, b, c, d):
        """Evaluate the Boolean function by looking up (a, b, c, d) in the truth table."""
        return truth_table[(a, b, c, d)]

    return f

### Demonstration

We call `random_constant_balanced()` to obtain a function, evaluate it on all 16 inputs, and display the resulting truth table.
By counting the number of `True` outputs we can verify which type was generated.

In [3]:
# All 16 possible four-bit input combinations (reused across demonstration cells)
all_inputs = list(itertools.product([False, True], repeat=4))

In [4]:
# Generate a single random constant-or-balanced function
func = random_constant_balanced()

# Evaluate the function on every possible input and store results
outputs = [func(a, b, c, d) for a, b, c, d in all_inputs]

# Count True outputs to identify the function type
true_count = sum(outputs)

# Classify based on the number of True outputs
if true_count == 0:
    detected_type = "Constant (always False)"
elif true_count == 16:
    detected_type = "Constant (always True)"
elif true_count == 8:
    detected_type = "Balanced"
else:
    detected_type = f"Unknown ({true_count} True outputs)"

print(f"Detected function type: {detected_type}")
print(f"True outputs: {true_count} / 16\n")

# Display the full truth table
print(f"{'a':>5}  {'b':>5}  {'c':>5}  {'d':>5}  |  {'f(a,b,c,d)':>10}")
print("-" * 46)
for (a, b, c, d), output in zip(all_inputs, outputs):
    print(f"{str(a):>5}  {str(b):>5}  {str(c):>5}  {str(d):>5}  |  {str(output):>10}")

Detected function type: Constant (always True)
True outputs: 16 / 16

    a      b      c      d  |  f(a,b,c,d)
----------------------------------------------
False  False  False  False  |        True
False  False  False   True  |        True
False  False   True  False  |        True
False  False   True   True  |        True
False   True  False  False  |        True
False   True  False   True  |        True
False   True   True  False  |        True
False   True   True   True  |        True
 True  False  False  False  |        True
 True  False  False   True  |        True
 True  False   True  False  |        True
 True  False   True   True  |        True
 True   True  False  False  |        True
 True   True  False   True  |        True
 True   True   True  False  |        True
 True   True   True   True  |        True


## Problem 2: Classical Testing for Function Type

### Background

[Deutsch's algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm) claims a quantum advantage over classical computation.
To appreciate that advantage, we must first establish the classical cost — i.e., the minimum number of times a classical algorithm must call `f` to determine its type with absolute certainty.

For a function `f` taking **four Boolean inputs**, there are $2^4 = 16$ possible input combinations.

| Function type | Condition |
|---|---|
| Constant | All 16 outputs are the same value (`True` or `False`) |
| Balanced | Exactly 8 outputs are `True` and 8 are `False` |

A classical algorithm cannot determine the type from a single call — the same output could come from either type.
The question is: **what is the minimum number of calls guaranteed to be sufficient?**

### Approach

The key insight is that we do **not** need to evaluate `f` on all 16 inputs.

- If two queries ever return **different values**, the function cannot be constant → return `"balanced"` immediately.
- If the same value is seen **9 times in a row**, the function cannot be balanced (a balanced function produces each value at most 8 times) → return `"constant"` immediately.

This gives us an early-exit strategy that needs **at most 9 calls** to reach a definitive answer.

In [5]:
def determine_constant_balanced(f):
    """
    Determine whether a Boolean function is constant or balanced.

    Queries f on successive distinct inputs and stops as soon as the type
    can be established with certainty.  At most 9 calls to f are ever needed.

    Parameters
    ----------
    f : callable
        A function accepting four Boolean positional arguments (a, b, c, d)
        and returning a single Boolean value.  The function is guaranteed to
        be either constant or balanced, as produced by random_constant_balanced.

    Returns
    -------
    str
        "constant" if f returns the same value for all 16 inputs, or
        "balanced" if f returns True for exactly 8 of the 16 inputs.
    """
    # All 16 four-bit input combinations in a fixed evaluation order
    all_inputs = list(itertools.product([False, True], repeat=4))

    # First query establishes the reference output value
    first_output = f(*all_inputs[0])

    # Query up to 8 more inputs (9 total); stop early if the type is clear
    # Reasoning: a balanced function has exactly 8 True and 8 False outputs,
    # so it cannot produce the same value 9 times — that would require 9 of one
    # value, leaving only 7 of the other, violating the balanced constraint.
    for inputs in all_inputs[1:9]:
        output = f(*inputs)

        # A differing output proves the function is not constant → balanced
        if output != first_output:
            return "balanced"

    # Nine consecutive identical outputs → impossible for balanced → constant
    return "constant"

## Problem 3: Quantum Oracles

## Problem 4: Deutsch's Algorithm with Qiskit

## Problem 5: Scaling to the Deutsch–Jozsa Algorithm